# RQ2: Robustness Analysis

**Research Question**: *Is the observed maturity structure robust to tool-specific bias, polyglot tooling effects, repository characteristics, and classification threshold choices?*

| Sub-question | Focus | Key Tests |
|---|---|---|
| **RQ2a** | Tool-specific bias | Chi-squared, Cramer's V, Kruskal-Wallis |
| **RQ2b** | Polyglot tooling effects | Spearman, Mann-Whitney U, partial correlation |
| **RQ2c** | Repo characteristic confounds | Spearman, partial correlation (ordinal logistic skipped — `statsmodels` unavailable in this run) |
| **RQ2d** | Classification threshold sensitivity | Threshold sweeps, Guttman CR/CS stability |

**Dataset**: 210 scored repositories (from `data/rq1_file_predictions.parquet`, notebook-13-style scoring), 1,046 **strict+W+ AI artifacts** at the artifact level (`discovery_step ∈ {tool_standard, shared_in_tool_folder, shared_in_root}` plus W+ exact-basename recoveries of nested tool files, matching RQ1), 768-dimensional embeddings (nomic-embed-text-v1.5). RQ2 analyzes the scored population; the L1 padding of whitelist-excluded repos (RQ1 population framing) is not applied here because RQ2 asks about robustness of the maturity structure among scored repos.

In [ ]:
import os
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"

import json
import pickle
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
import plotly.io as pio
from plotly.subplots import make_subplots
from scipy.stats import chi2_contingency, kruskal, mannwhitneyu, spearmanr
from sklearn.metrics.pairwise import cosine_similarity

pio.renderers.default = "notebook"
warnings.filterwarnings("ignore")

In [ ]:
# Paths
DATA_DIR = Path("../../data")
OUTPUT_DIR = Path("../../output")
FIGURES_DIR = Path("figures")
FIGURES_DIR.mkdir(exist_ok=True)

# Constants
CATEGORY_NAMES = [
    "agents", "commands", "flows", "rules", "skills",
    "architecture", "code-style", "configuration", "session-logs",
]
CATEGORY_TO_LEVEL_INT = {
    "rules": 2, "configuration": 2, "architecture": 2, "code-style": 2,
    "agents": 3, "commands": 3, "skills": 3,
    "flows": 4, "session-logs": 4,
}
LEVEL_COLORS = {1: "#6b7280", 2: "#3b82f6", 3: "#f97316", 4: "#22c55e"}
LEVEL_LABELS = {1: "L1 Ad Hoc", 2: "L2 Grounded", 3: "L3 Agent-Augmented", 4: "L4 Orchestration"}

# Strict AI-artifact filter (same WL_STEPS definition as notebooks 13/14),
# extended with the W+ exact-basename recoveries (nested CLAUDE.md/AGENTS.md,
# mcp configs, ...) to match RQ1. rq1_repo_scores.csv is already strict+W+
# (rebuilt by RQ1 with STRICT_MODE + WL_PLUS); the artifact-level frames and
# the position-aligned embeddings need the same mask applied here.
STRICT_MODE = True
WL_STEPS = {"tool_standard", "shared_in_tool_folder", "shared_in_root"}
WL_PLUS = True

import sys
sys.path.insert(0, str(Path("../..").resolve()))
from src.artifact_filtering import load_protected_patterns
PROTECTED_EXACT, _, _ = load_protected_patterns(str(Path("../../Artifacts").resolve()))

def wl_mask(df, name_col="artifact_name"):
    m = df["discovery_step"].isin(WL_STEPS)
    if WL_PLUS:
        m |= df[name_col].str.lower().isin(PROTECTED_EXACT)
    return m

# Load data
repo_scores_df = pd.read_csv(DATA_DIR / "rq1_repo_scores.csv")
metadata_df = pd.read_csv(DATA_DIR / "filtered_common_metadata.csv")
ai_metadata_df = pd.read_csv(DATA_DIR / "filtered_ai_tools_metadata.csv")

emb_data = np.load(DATA_DIR / "filtered_common_embeddings.npz")
emb_key = "embeddings" if "embeddings" in emb_data.files else emb_data.files[0]
embeddings = emb_data[emb_key]

if STRICT_MODE:
    assert len(metadata_df) == len(embeddings), (
        f"metadata/embeddings misaligned before strict mask: "
        f"{len(metadata_df)} vs {len(embeddings)}"
    )
    strict_mask = wl_mask(metadata_df).to_numpy()
    metadata_df = metadata_df[strict_mask].reset_index(drop=True)
    embeddings = embeddings[strict_mask]
    ai_metadata_df = ai_metadata_df[wl_mask(ai_metadata_df)].reset_index(drop=True)

with open(DATA_DIR / "classification_model.pkl", "rb") as f:
    classification_model = pickle.load(f)

# category_embeddings is a dict {name: ndarray(768,)} — stack into matrix
template_category_names = classification_model["category_names"]
cat_emb_dict = classification_model["category_embeddings"]
template_embeddings = np.stack([cat_emb_dict[c] for c in template_category_names])

# Derived columns
cat_cols = [f"cat_{c}" for c in CATEGORY_NAMES]
repo_scores_df["n_categories"] = (repo_scores_df[cat_cols] > 0).sum(axis=1)
repo_scores_df["tools_list"] = repo_scores_df["tools"].apply(
    lambda x: [t.strip() for t in str(x).split(", ") if t.strip() and t.strip() != "nan"]
    if pd.notna(x) and str(x).strip() else []
)

# AI-tools subset
ai_repos = set(ai_metadata_df["repo_name"].unique())
ai_scores_df = repo_scores_df[repo_scores_df["full_repo_name"].isin(ai_repos)].copy()

print(f"Full dataset: {len(repo_scores_df)} repos, {len(metadata_df)} artifacts")
print(f"AI-tools subset: {len(ai_scores_df)} repos")
print(f"Embeddings: {embeddings.shape}")
print(f"Template embeddings: {template_embeddings.shape} ({len(template_category_names)} categories)")
print(f"Level distribution:")
print(repo_scores_df["level"].value_counts().sort_index())

In [ ]:
def save_fig(fig, filename, width=1200, height=600):
    stem = Path(filename).stem
    for ext, kwargs in ((".png", {"scale": 2}), (".pdf", {})):
        path = FIGURES_DIR / f"{stem}{ext}"
        try:
            fig.write_image(str(path), width=width, height=height, **kwargs)
            print(f"Saved: {path}")
        except Exception as e:
            print(f"Could not save {path}: {e}")

def cliff_delta(x, y):
    x, y = np.asarray(x, dtype=float), np.asarray(y, dtype=float)
    if len(x) == 0 or len(y) == 0:
        return 0.0
    more = np.sum(x[:, None] > y[None, :])
    less = np.sum(x[:, None] < y[None, :])
    return (more - less) / (len(x) * len(y))

def cliff_delta_label(d):
    d = abs(d)
    if d < 0.147: return "negligible"
    elif d < 0.33: return "small"
    elif d < 0.474: return "medium"
    else: return "large"

def eta_squared(h_stat, n, k):
    return (h_stat - k + 1) / (n - k)

def cramers_v(table):
    chi2 = chi2_contingency(table)[0]
    n = table.values.sum() if hasattr(table, "values") else table.sum()
    k = min(table.shape) - 1
    return np.sqrt(chi2 / (n * k)) if n * k > 0 else 0.0

def partial_spearman(x, y, z):
    r_xy, _ = spearmanr(x, y)
    r_xz, _ = spearmanr(x, z)
    r_yz, _ = spearmanr(y, z)
    num = r_xy - r_xz * r_yz
    den = np.sqrt((1 - r_xz**2) * (1 - r_yz**2))
    return num / den if den > 0 else 0.0

def compute_guttman(levels, has_l2, has_l3, has_l4):
    """Compute Guttman CR and CS from level assignments and evidence presence."""
    n = len(levels)
    if n == 0:
        return 0.0, 0.0
    observed = np.column_stack([has_l2.astype(int), has_l3.astype(int), has_l4.astype(int)])
    expected = np.zeros((n, 3), dtype=int)
    for i, lv in enumerate(levels):
        if lv >= 2: expected[i, 0] = 1
        if lv >= 3: expected[i, 1] = 1
        if lv >= 4: expected[i, 2] = 1
    errors = np.sum(expected != observed)
    total = n * 3
    CR = 1 - errors / total
    modal_fracs = []
    for j in range(3):
        col = observed[:, j]
        modal_fracs.append(max(np.sum(col == 0), np.sum(col == 1)) / n)
    MMR = np.mean(modal_fracs)
    CS = (CR - MMR) / (1 - MMR) if MMR < 1 else 0.0
    return CR, CS

## RQ2a: Tool-Specific Bias

*"Do category distributions differ by AI tool?"*

If the maturity structure is genuine, it should hold across tools — repos using Cursor, Claude Code, or GitHub Copilot should show similar maturity distributions after controlling for sample size.

In [ ]:
# Explode tools to one row per tool per repo
tool_exploded = repo_scores_df.explode("tools_list").reset_index(drop=True)
tool_exploded = tool_exploded[tool_exploded["tools_list"].apply(lambda x: isinstance(x, str) and len(x) > 0)]
tool_exploded = tool_exploded.rename(columns={"tools_list": "tool"})

# Filter to major tools (n >= 20), exclude "shared"
tool_counts = tool_exploded["tool"].value_counts()
major_tools = [t for t in tool_counts[tool_counts >= 20].index if t != "shared"]
tool_major_df = tool_exploded[tool_exploded["tool"].isin(major_tools)].copy()

print("Repos per tool (all):")
print(tool_counts.to_string())
print(f"\nMajor tools (n >= 20): {major_tools}")
for t in major_tools:
    print(f"  {t}: n={tool_counts[t]}")

In [ ]:
# Chart 2a.1: Level distribution by tool
level_by_tool = pd.crosstab(tool_major_df["tool"], tool_major_df["level"], normalize="index") * 100

fig = go.Figure()
for level in sorted(repo_scores_df["level"].unique()):
    if level in level_by_tool.columns:
        fig.add_trace(go.Bar(
            name=LEVEL_LABELS[level], x=level_by_tool.index,
            y=level_by_tool[level], marker_color=LEVEL_COLORS[level],
            text=[f"{v:.0f}%" for v in level_by_tool[level]],
            textposition="auto",
        ))
fig.update_layout(
    title="RQ2a: Maturity Level Distribution by AI Tool",
    xaxis_title="AI Tool", yaxis_title="% of Repos",
    barmode="group", height=450, width=700,
    legend=dict(title="Level"),
)
fig.show()

# Chi-squared test
contingency = pd.crosstab(tool_major_df["tool"], tool_major_df["level"])
chi2, p_chi2, dof, _ = chi2_contingency(contingency)
v = cramers_v(contingency)
print(f"Chi-squared test of independence: chi2={chi2:.2f}, df={dof}, p={p_chi2:.4f}")
print(f"Cramer's V = {v:.3f}")

save_fig(fig, "rq2a_level_by_tool.png", width=700, height=450)

In [ ]:
# Chart 2a.2: Category profiles by tool (heatmap)
cat_profiles = {}
for tool in major_tools:
    tool_repos = tool_major_df[tool_major_df["tool"] == tool]
    profile = {}
    for cat in CATEGORY_NAMES:
        col = f"cat_{cat}"
        if col in tool_repos.columns:
            profile[cat] = (tool_repos[col] > 0).mean() * 100
    cat_profiles[tool] = profile

profile_df = pd.DataFrame(cat_profiles).T
# Order columns by level
level_order = sorted(CATEGORY_NAMES, key=lambda c: (CATEGORY_TO_LEVEL_INT[c], c))
profile_df = profile_df[level_order]

fig = go.Figure(data=go.Heatmap(
    z=profile_df.values, x=profile_df.columns, y=profile_df.index,
    colorscale="Blues", text=[[f"{v:.0f}%" for v in row] for row in profile_df.values],
    texttemplate="%{text}", textfont={"size": 11},
    colorbar=dict(title="% repos"),
))
fig.update_layout(
    title="RQ2a: Category Adoption Profiles by AI Tool",
    xaxis_title="Category", yaxis_title="Tool",
    height=350, width=800,
)
fig.show()

# Kruskal-Wallis per category with Bonferroni correction
alpha_bonferroni = 0.05 / len(CATEGORY_NAMES)
print(f"\nKruskal-Wallis per category (Bonferroni alpha={alpha_bonferroni:.4f}):")
for cat in CATEGORY_NAMES:
    col = f"cat_{cat}"
    groups = [tool_major_df[tool_major_df["tool"] == t][col].values for t in major_tools]
    if all(len(g) > 0 for g in groups):
        h, p = kruskal(*groups)
        sig = "***" if p < alpha_bonferroni else ""
        print(f"  {cat:15s}: H={h:.2f}, p={p:.4f} {sig}")

save_fig(fig, "rq2a_category_by_tool.png", width=800, height=350)

In [ ]:
# Chart 2a.3: Category breadth by tool (box plot)
fig = go.Figure()
for tool in major_tools:
    data = tool_major_df[tool_major_df["tool"] == tool]["n_categories"]
    fig.add_trace(go.Box(y=data, name=tool, boxmean=True))
fig.update_layout(
    title="RQ2a: Category Breadth by AI Tool",
    yaxis_title="Number of Categories", height=400, width=600,
)
fig.show()

# Pairwise Mann-Whitney U + Cliff's delta
print("Pairwise Mann-Whitney U + Cliff's delta (breadth):")
for i, t1 in enumerate(major_tools):
    for t2 in major_tools[i+1:]:
        d1 = tool_major_df[tool_major_df["tool"] == t1]["n_categories"].values
        d2 = tool_major_df[tool_major_df["tool"] == t2]["n_categories"].values
        u, p = mannwhitneyu(d1, d2, alternative="two-sided")
        cd = cliff_delta(d1, d2)
        print(f"  {t1} vs {t2}: U={u:.0f}, p={p:.4f}, delta={cd:.3f} ({cliff_delta_label(cd)})")

In [ ]:
# Statistical summary table
summary_rows = []
for tool in major_tools:
    t_df = tool_major_df[tool_major_df["tool"] == tool]
    n = len(t_df)
    med_level = t_df["level"].median()
    mean_breadth = t_df["n_categories"].mean()
    pct_l2 = (t_df["level"] == 2).mean() * 100
    pct_l3 = (t_df["level"] == 3).mean() * 100
    pct_l4 = (t_df["level"] == 4).mean() * 100
    summary_rows.append({
        "tool": tool, "n": n, "median_level": med_level,
        "mean_breadth": f"{mean_breadth:.2f}",
        "L2%": f"{pct_l2:.1f}", "L3%": f"{pct_l3:.1f}", "L4%": f"{pct_l4:.1f}",
    })
summary_df = pd.DataFrame(summary_rows)
print("RQ2a Summary:")
print(summary_df.to_string(index=False))

In [ ]:
# Robustness: Repeat on AI-tools subset
ai_tool_exploded = ai_scores_df.explode("tools_list").reset_index(drop=True)
ai_tool_exploded = ai_tool_exploded[ai_tool_exploded["tools_list"].apply(lambda x: isinstance(x, str) and len(x) > 0)]
ai_tool_exploded = ai_tool_exploded.rename(columns={"tools_list": "tool"})
ai_tool_counts = ai_tool_exploded["tool"].value_counts()
ai_major = [t for t in ai_tool_counts[ai_tool_counts >= 15].index if t != "shared"]

if len(ai_major) >= 2:
    ai_tool_sub = ai_tool_exploded[ai_tool_exploded["tool"].isin(ai_major)]
    contingency_ai = pd.crosstab(ai_tool_sub["tool"], ai_tool_sub["level"])
    chi2_ai, p_ai, dof_ai, _ = chi2_contingency(contingency_ai)
    v_ai = cramers_v(contingency_ai)
    print(f"AI-tools subset robustness (n={len(ai_tool_sub)}, tools={ai_major}):")
    print(f"  Chi-squared: chi2={chi2_ai:.2f}, df={dof_ai}, p={p_ai:.4f}, Cramer's V={v_ai:.3f}")
else:
    print("Not enough major tools in AI-tools subset for robustness check")

## RQ2b: Polyglot Tooling

*"Do multi-tool repos reach higher maturity?"*

If maturity reflects genuine practice depth, repos using multiple AI tools might show higher maturity — but this could also be a confound (more tools → more config files → higher level by construction).

In [ ]:
# Group n_tools
def ntools_group(n):
    if n == 0: return "0"
    elif n == 1: return "1"
    elif n == 2: return "2"
    else: return "3+"

repo_scores_df["ntools_group"] = repo_scores_df["n_tools"].apply(ntools_group)
group_order = ["0", "1", "2", "3+"]

print("Repos by n_tools group:")
for g in group_order:
    n = (repo_scores_df["ntools_group"] == g).sum()
    print(f"  {g}: n={n}")

In [ ]:
# Chart 2b.1: Level distribution by n_tools group
level_by_ntools = pd.crosstab(
    repo_scores_df["ntools_group"], repo_scores_df["level"], normalize="index"
) * 100
level_by_ntools = level_by_ntools.reindex(group_order)

fig = go.Figure()
for level in sorted(repo_scores_df["level"].unique()):
    if level in level_by_ntools.columns:
        fig.add_trace(go.Bar(
            name=LEVEL_LABELS[level], x=level_by_ntools.index,
            y=level_by_ntools[level], marker_color=LEVEL_COLORS[level],
            text=[f"{v:.0f}%" for v in level_by_ntools[level]],
            textposition="auto",
        ))
fig.update_layout(
    title="RQ2b: Maturity Level Distribution by Number of AI Tools",
    xaxis_title="Number of AI Tools", yaxis_title="% of Repos",
    barmode="group", height=450, width=700,
    legend=dict(title="Level"),
)
fig.show()
save_fig(fig, "rq2b_level_by_ntools.png", width=700, height=450)

In [ ]:
# Spearman correlation: n_tools vs level
rho, p_rho = spearmanr(repo_scores_df["n_tools"], repo_scores_df["level"])
print(f"Spearman(n_tools, level): rho={rho:.3f}, p={p_rho:.4e}")

# Box plot: level by n_tools
fig = go.Figure()
for g in group_order:
    data = repo_scores_df[repo_scores_df["ntools_group"] == g]["level"]
    fig.add_trace(go.Box(y=data, name=f"{g} tools", boxmean=True))
fig.update_layout(
    title=f"RQ2b: Maturity Level by Tool Count (Spearman rho={rho:.3f}, p={p_rho:.2e})",
    yaxis_title="Maturity Level", height=400, width=600,
)
fig.show()

# Mann-Whitney: 1-tool vs multi-tool
single = repo_scores_df[repo_scores_df["n_tools"] == 1]["level"].values
multi = repo_scores_df[repo_scores_df["n_tools"] >= 2]["level"].values
if len(single) > 0 and len(multi) > 0:
    u, p_u = mannwhitneyu(single, multi, alternative="two-sided")
    cd = cliff_delta(multi, single)
    print(f"Mann-Whitney U (1-tool vs multi-tool): U={u:.0f}, p={p_u:.4f}")
    print(f"Cliff's delta = {cd:.3f} ({cliff_delta_label(cd)})")

In [ ]:
# Chart 2b.3: Category breadth vs n_tools
rho_b, p_b = spearmanr(repo_scores_df["n_tools"], repo_scores_df["n_categories"])
fig = go.Figure()
for g in group_order:
    data = repo_scores_df[repo_scores_df["ntools_group"] == g]["n_categories"]
    fig.add_trace(go.Box(y=data, name=f"{g} tools", boxmean=True))
fig.update_layout(
    title=f"RQ2b: Category Breadth by Tool Count (Spearman rho={rho_b:.3f})",
    yaxis_title="Number of Categories", height=400, width=600,
)
fig.show()
print(f"Spearman(n_tools, n_categories): rho={rho_b:.3f}, p={p_b:.4e}")

In [ ]:
# Confound control: partial Spearman (n_tools vs level, controlling for artifact_count)
r_partial = partial_spearman(
    repo_scores_df["n_tools"].values,
    repo_scores_df["level"].values,
    repo_scores_df["artifact_count"].values,
)
print(f"Partial Spearman(n_tools, level | artifact_count): r={r_partial:.3f}")
print(f"  Zero-order Spearman(n_tools, level): rho={rho:.3f}")
print(f"  Attenuation: {abs(rho) - abs(r_partial):.3f}")

In [ ]:
# Chart 2b.4: 100% stacked bar by n_tools group
fig = go.Figure()
for level in sorted(repo_scores_df["level"].unique()):
    if level in level_by_ntools.columns:
        fig.add_trace(go.Bar(
            name=LEVEL_LABELS[level], x=level_by_ntools.index,
            y=level_by_ntools[level], marker_color=LEVEL_COLORS[level],
        ))
fig.update_layout(
    title="RQ2b: Maturity Composition by Tool Count",
    xaxis_title="Number of AI Tools", yaxis_title="% of Repos",
    barmode="stack", height=450, width=700,
    legend=dict(title="Level"),
)
fig.show()
save_fig(fig, "rq2b_transition_by_ntools.png", width=700, height=450)

In [ ]:
# Robustness: AI-tools subset (excludes n_tools=0 by definition)
ai_scores_df["ntools_group"] = ai_scores_df["n_tools"].apply(ntools_group)
rho_ai, p_ai = spearmanr(ai_scores_df["n_tools"], ai_scores_df["level"])
r_partial_ai = partial_spearman(
    ai_scores_df["n_tools"].values,
    ai_scores_df["level"].values,
    ai_scores_df["artifact_count"].values,
)
print(f"AI-tools subset (n={len(ai_scores_df)}):")
print(f"  Spearman(n_tools, level): rho={rho_ai:.3f}, p={p_ai:.4e}")
print(f"  Partial Spearman(n_tools, level | artifact_count): r={r_partial_ai:.3f}")

## RQ2c: Repo Characteristic Confounds

*"Do repo characteristics confound maturity?"*

We test whether repository size, age, activity, or primary language predict maturity levels independently of AI tool adoption patterns.

In [ ]:
# Load repo metrics from output directories
metrics_records = []
for _, row in repo_scores_df.iterrows():
    org = str(row["org_name"])
    repo = str(row["repo_name"])
    path = OUTPUT_DIR / org / repo / f"{repo}_repo_metrics.json"
    if path.exists():
        with open(path) as f:
            m = json.load(f)
        m["full_repo_name"] = row["full_repo_name"]
        metrics_records.append(m)

metrics_df = pd.DataFrame(metrics_records)
print(f"Loaded metrics for {len(metrics_df)} / {len(repo_scores_df)} repos")

# Derive features
metrics_df["n_languages"] = metrics_df["languages"].apply(lambda x: len(x) if isinstance(x, dict) else 0)
metrics_df["primary_language"] = metrics_df["languages"].apply(
    lambda x: max(x, key=x.get) if isinstance(x, dict) and len(x) > 0 else "Unknown"
)
metrics_df["log_total_lines"] = np.log1p(pd.to_numeric(metrics_df["total_lines"], errors="coerce").fillna(0))
metrics_df["log_total_files"] = np.log1p(pd.to_numeric(metrics_df["total_files"], errors="coerce").fillna(0))
metrics_df["commit_velocity"] = pd.to_numeric(metrics_df["commits_last_year"], errors="coerce").fillna(0) / 365.0

# Repo age — compute via total_seconds to avoid .dt accessor issues
first_dt = pd.to_datetime(metrics_df["first_commit_date"], errors="coerce", utc=True)
last_dt = pd.to_datetime(metrics_df["last_commit_date"], errors="coerce", utc=True)
delta = last_dt - first_dt
metrics_df["repo_age_days"] = delta.apply(lambda x: x.total_seconds() / 86400 if pd.notna(x) else 0)

# Merge with repo scores
confound_df = repo_scores_df.merge(metrics_df, on="full_repo_name", how="inner")
print(f"Merged dataset: {len(confound_df)} repos with both scores and metrics")
print(f"Primary languages: {confound_df['primary_language'].nunique()} unique")

In [ ]:
# Chart 2c.1: Size metrics by level (2x2 box plots)
size_features = [
    ("log_total_lines", "log(Total Lines)"),
    ("log_total_files", "log(Total Files)"),
    ("total_commits", "Total Commits"),
    ("total_authors", "Total Authors"),
]
fig = make_subplots(rows=2, cols=2, subplot_titles=[t for _, t in size_features])
for idx, (feat, title) in enumerate(size_features):
    row, col = idx // 2 + 1, idx % 2 + 1
    for level in sorted(confound_df["level"].unique()):
        data = confound_df[confound_df["level"] == level][feat].dropna()
        fig.add_trace(
            go.Box(y=data, name=LEVEL_LABELS[level], marker_color=LEVEL_COLORS[level],
                   legendgroup=str(level), showlegend=(idx == 0)),
            row=row, col=col,
        )
    # Kruskal-Wallis (guard against constant data)
    groups = [confound_df[confound_df["level"] == lv][feat].dropna().values
              for lv in sorted(confound_df["level"].unique())]
    groups = [g for g in groups if len(g) > 0]
    all_vals = np.concatenate(groups) if groups else np.array([])
    if len(groups) >= 2 and len(np.unique(all_vals)) > 1:
        h, p = kruskal(*groups)
        xr = "x domain" if idx == 0 else f"x{idx+1} domain"
        yr = "y domain" if idx == 0 else f"y{idx+1} domain"
        fig.add_annotation(text=f"H={h:.1f}, p={p:.3f}", xref=xr, yref=yr,
                           x=0.5, y=1.12, showarrow=False, font=dict(size=10))

fig.update_layout(height=600, width=900, title="RQ2c: Repository Size Metrics by Maturity Level",
                  showlegend=True)
fig.show()
save_fig(fig, "rq2c_size_by_level.png", width=900, height=600)

In [ ]:
# Chart 2c.2: Temporal metrics by level (1x2)
temp_features = [
    ("repo_age_days", "Repo Age (days)"),
    ("commit_velocity", "Commit Velocity (commits/day)"),
]
fig = make_subplots(rows=1, cols=2, subplot_titles=[t for _, t in temp_features])
for idx, (feat, title) in enumerate(temp_features):
    col = idx + 1
    for level in sorted(confound_df["level"].unique()):
        data = confound_df[confound_df["level"] == level][feat].dropna()
        fig.add_trace(
            go.Box(y=data, name=LEVEL_LABELS[level], marker_color=LEVEL_COLORS[level],
                   legendgroup=str(level), showlegend=(idx == 0)),
            row=1, col=col,
        )
    groups = [confound_df[confound_df["level"] == lv][feat].dropna().values
              for lv in sorted(confound_df["level"].unique())]
    groups = [g for g in groups if len(g) > 0]
    all_vals = np.concatenate(groups) if groups else np.array([])
    if len(groups) >= 2 and len(np.unique(all_vals)) > 1:
        h, p = kruskal(*groups)
        xr = "x domain" if idx == 0 else f"x{idx+1} domain"
        yr = "y domain" if idx == 0 else f"y{idx+1} domain"
        fig.add_annotation(text=f"H={h:.1f}, p={p:.3f}", xref=xr, yref=yr,
                           x=0.5, y=1.12, showarrow=False, font=dict(size=10))

fig.update_layout(height=400, width=800, title="RQ2c: Temporal Metrics by Maturity Level", showlegend=True)
fig.show()
save_fig(fig, "rq2c_temporal_by_level.png", width=800, height=400)

In [ ]:
# Chart 2c.3: Primary language vs level
lang_counts = confound_df["primary_language"].value_counts()
major_langs = lang_counts[lang_counts >= 10].index.tolist()
lang_df = confound_df[confound_df["primary_language"].isin(major_langs)].copy()

level_by_lang = pd.crosstab(lang_df["primary_language"], lang_df["level"], normalize="index") * 100

fig = go.Figure()
for level in sorted(lang_df["level"].unique()):
    if level in level_by_lang.columns:
        fig.add_trace(go.Bar(
            name=LEVEL_LABELS[level], x=level_by_lang.index,
            y=level_by_lang[level], marker_color=LEVEL_COLORS[level],
        ))
fig.update_layout(
    title="RQ2c: Maturity Level by Primary Language (n >= 10)",
    xaxis_title="Primary Language", yaxis_title="% of Repos",
    barmode="group", height=450, width=800,
)
fig.show()

# Chi-squared
contingency_lang = pd.crosstab(lang_df["primary_language"], lang_df["level"])
chi2_l, p_l, dof_l, _ = chi2_contingency(contingency_lang)
v_l = cramers_v(contingency_lang)
print(f"Chi-squared (language vs level): chi2={chi2_l:.2f}, df={dof_l}, p={p_l:.4f}, Cramer's V={v_l:.3f}")

save_fig(fig, "rq2c_language_by_level.png", width=800, height=450)

In [ ]:
# Spearman correlations table
numeric_features = [
    ("total_commits", "Commits"),
    ("total_authors", "Authors"),
    ("log_total_lines", "log(Lines)"),
    ("log_total_files", "log(Files)"),
    ("repo_age_days", "Repo Age"),
    ("commit_velocity", "Commit Velocity"),
    ("n_languages", "N Languages"),
    ("n_tools", "N Tools"),
]
alpha_bonferroni_c = 0.05 / len(numeric_features)

print(f"Spearman correlations with maturity level (Bonferroni alpha={alpha_bonferroni_c:.4f}):")
print(f"{'Feature':20s} {'rho':>8s} {'p-value':>12s} {'Sig':>5s}")
print("-" * 50)
corr_results = []
for feat, label in numeric_features:
    valid = confound_df[[feat, "level"]].dropna()
    if len(valid) > 5:
        rho_f, p_f = spearmanr(valid[feat], valid["level"])
        sig = "***" if p_f < alpha_bonferroni_c else ("*" if p_f < 0.05 else "")
        print(f"{label:20s} {rho_f:8.3f} {p_f:12.4e} {sig:>5s}")
        corr_results.append({"feature": feat, "label": label, "rho": rho_f, "p": p_f})

corr_results_df = pd.DataFrame(corr_results)

In [ ]:
# Cliff's delta: pairwise L2 vs L3, L2 vs L4, L3 vs L4
level_pairs = [(2, 3), (2, 4), (3, 4)]
print("Cliff's delta (pairwise level comparisons):")
print(f"{'Feature':20s}", end="")
for l1, l2 in level_pairs:
    print(f"  {'L'+str(l1)+' vs L'+str(l2):>15s}", end="")
print()
print("-" * 70)
for feat, label in numeric_features:
    print(f"{label:20s}", end="")
    for l1, l2 in level_pairs:
        d1 = confound_df[confound_df["level"] == l1][feat].dropna().values
        d2 = confound_df[confound_df["level"] == l2][feat].dropna().values
        if len(d1) > 0 and len(d2) > 0:
            cd = cliff_delta(d2, d1)
            print(f"  {cd:7.3f} ({cliff_delta_label(cd)[:3]:>3s})", end="")
        else:
            print(f"  {'N/A':>15s}", end="")
    print()

In [ ]:
# Chart 2c.4: Correlation heatmap
targets = ["level", "n_categories", "artifact_count"]
feat_names = [f for f, _ in numeric_features]
feat_labels = [l for _, l in numeric_features]

corr_matrix = np.zeros((len(feat_names), len(targets)))
for i, feat in enumerate(feat_names):
    for j, target in enumerate(targets):
        valid = confound_df[[feat, target]].dropna()
        if len(valid) > 5:
            rho_val, _ = spearmanr(valid[feat], valid[target])
            corr_matrix[i, j] = rho_val

fig = go.Figure(data=go.Heatmap(
    z=corr_matrix, x=targets, y=feat_labels,
    colorscale="RdBu_r", zmid=0, zmin=-0.5, zmax=0.5,
    text=[[f"{v:.2f}" for v in row] for row in corr_matrix],
    texttemplate="%{text}", textfont={"size": 12},
    colorbar=dict(title="Spearman rho"),
))
fig.update_layout(
    title="RQ2c: Spearman Correlations — Repo Features vs Maturity Indicators",
    height=450, width=600,
)
fig.show()
save_fig(fig, "rq2c_correlation_heatmap.png", width=600, height=450)

In [ ]:
# Partial correlations: control for artifact_count and n_tools
print("Partial Spearman correlations (controlling for artifact_count):")
for feat, label in numeric_features:
    if feat == "n_tools":
        continue
    valid = confound_df[[feat, "level", "artifact_count"]].dropna()
    if len(valid) > 5:
        r_p = partial_spearman(valid[feat].values, valid["level"].values, valid["artifact_count"].values)
        rho_raw, _ = spearmanr(valid[feat], valid["level"])
        print(f"  {label:20s}: raw={rho_raw:.3f}, partial={r_p:.3f}, change={abs(rho_raw)-abs(r_p):+.3f}")

print("\nPartial Spearman correlations (controlling for n_tools):")
for feat, label in numeric_features:
    if feat == "n_tools":
        continue
    valid = confound_df[[feat, "level", "n_tools"]].dropna()
    if len(valid) > 5:
        r_p = partial_spearman(valid[feat].values, valid["level"].values, valid["n_tools"].values)
        rho_raw, _ = spearmanr(valid[feat], valid["level"])
        print(f"  {label:20s}: raw={rho_raw:.3f}, partial={r_p:.3f}, change={abs(rho_raw)-abs(r_p):+.3f}")

In [ ]:
# Ordinal logistic regression
try:
    from statsmodels.miscmodels.ordinal_model import OrderedModel

    # Prepare features
    olr_features = ["log_total_lines", "total_commits", "total_authors",
                     "repo_age_days", "commit_velocity", "n_tools"]
    olr_df = confound_df[olr_features + ["level"]].dropna()

    # Standardize features for comparable coefficients
    from sklearn.preprocessing import StandardScaler
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(olr_df[olr_features])
    X_df = pd.DataFrame(X_scaled, columns=olr_features)
    X_df["level"] = olr_df["level"].values

    model = OrderedModel(X_df["level"], X_df[olr_features], distr="logit")
    result = model.fit(method="bfgs", disp=False)
    print("Ordinal Logistic Regression: level ~ repo features")
    print(result.summary())

    # Extract coefficients for forest plot
    olr_coefs = []
    for feat in olr_features:
        idx = list(result.params.index).index(feat) if feat in result.params.index else None
        if idx is not None:
            coef = result.params[feat]
            se = result.bse[feat]
            olr_coefs.append({"feature": feat, "coef": coef, "se": se,
                              "odds_ratio": np.exp(coef),
                              "ci_low": coef - 1.96*se, "ci_high": coef + 1.96*se})
    olr_coefs_df = pd.DataFrame(olr_coefs)
    print(f"\nPseudo R-squared: {result.prsquared:.4f}")
except ImportError:
    print("statsmodels not available for ordinal logistic regression")
    olr_coefs_df = pd.DataFrame()
except Exception as e:
    print(f"Ordinal logistic regression failed: {e}")
    olr_coefs_df = pd.DataFrame()

In [ ]:
# Chart 2c.5: Effect size summary (forest/dot-whisker plot)
if len(olr_coefs_df) > 0:
    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=olr_coefs_df["coef"], y=olr_coefs_df["feature"],
        error_x=dict(type="data",
                     symmetric=False,
                     array=olr_coefs_df["ci_high"] - olr_coefs_df["coef"],
                     arrayminus=olr_coefs_df["coef"] - olr_coefs_df["ci_low"]),
        mode="markers", marker=dict(size=10, color="#3b82f6"),
    ))
    fig.add_vline(x=0, line_dash="dash", line_color="gray")
    fig.update_layout(
        title="RQ2c: Ordinal Logistic Regression Coefficients (standardized)",
        xaxis_title="Coefficient (log-odds)", height=400, width=700,
    )
    fig.show()
    save_fig(fig, "rq2c_effect_summary.png", width=700, height=400)
else:
    # Fallback: plot Spearman correlations as effect sizes
    fig = go.Figure()
    fig.add_trace(go.Bar(
        x=corr_results_df["rho"], y=corr_results_df["label"],
        orientation="h", marker_color="#3b82f6",
    ))
    fig.add_vline(x=0, line_dash="dash", line_color="gray")
    fig.update_layout(
        title="RQ2c: Spearman Correlations with Maturity Level",
        xaxis_title="Spearman rho", height=400, width=700,
    )
    fig.show()
    save_fig(fig, "rq2c_effect_summary.png", width=700, height=400)

In [ ]:
# Robustness: repeat key correlations on AI-tools subset
ai_confound = ai_scores_df.merge(metrics_df, on="full_repo_name", how="inner")
print(f"AI-tools subset confound analysis (n={len(ai_confound)}):")
for feat, label in numeric_features[:6]:
    valid = ai_confound[[feat, "level"]].dropna()
    if len(valid) > 5:
        rho_ai, p_ai = spearmanr(valid[feat], valid["level"])
        print(f"  {label:20s}: rho={rho_ai:.3f}, p={p_ai:.4e}")

## RQ2d: Sensitivity Analysis

*"How stable are classifications across the similarity threshold?"*

We sweep three threshold parameters to test whether maturity level assignments are robust to classification confidence requirements. If levels are genuinely structured, moderate threshold changes should not dramatically alter the distribution.

| Approach | Sweep Range | What it varies |
|---|---|---|
| A: Min cosine similarity | 0.00 – 0.40 | Files below threshold become unclassified |
| B: Min margin filter | 0.00 – 0.15 | Ambiguous files (small top1-top2 gap) excluded |
| C: HYBRID_THRESHOLD | 0.01 – 0.10 | Secondary evidence breadth |

In [ ]:
# Compute cosine similarity: all files vs templates
# Align metadata with embeddings (only rows with has_embedding=True)
meta_with_emb = metadata_df[metadata_df["has_embedding"] == True].reset_index(drop=True)

# Verify alignment
assert len(meta_with_emb) == embeddings.shape[0], (
    f"Metadata rows ({len(meta_with_emb)}) != embedding rows ({embeddings.shape[0]})"
)

# Cosine similarity: (N, 768) vs (K, 768) -> (N, K)
cosine_scores = cosine_similarity(embeddings, template_embeddings)
print(f"Cosine similarity matrix: {cosine_scores.shape}")
print(f"  (files={cosine_scores.shape[0]}, categories={cosine_scores.shape[1]})")
print(f"  Template categories: {template_category_names}")

# Per-file stats
top_indices = np.argmax(cosine_scores, axis=1)
top_scores = np.max(cosine_scores, axis=1)
sorted_scores = np.sort(cosine_scores, axis=1)[:, ::-1]
margins = sorted_scores[:, 0] - sorted_scores[:, 1]
primary_categories = [template_category_names[i] for i in top_indices]

meta_with_emb["primary_category"] = primary_categories
meta_with_emb["top_score"] = top_scores
meta_with_emb["margin"] = margins
meta_with_emb["assigned_level"] = meta_with_emb["primary_category"].map(CATEGORY_TO_LEVEL_INT)

print(f"\nScore statistics:")
print(f"  Top-1 score: mean={top_scores.mean():.3f}, median={np.median(top_scores):.3f}")
print(f"  Margin: mean={margins.mean():.3f}, median={np.median(margins):.3f}")

In [ ]:
# Chart 2d.1: Score distributions
fig = make_subplots(rows=1, cols=2, subplot_titles=["Top-1 Cosine Similarity", "Classification Margin (top1 - top2)"])

fig.add_trace(go.Histogram(x=top_scores, nbinsx=50, marker_color="#3b82f6", name="Top-1 Score"), row=1, col=1)
fig.add_trace(go.Histogram(x=margins, nbinsx=50, marker_color="#f97316", name="Margin"), row=1, col=2)

fig.update_layout(height=400, width=900, title="RQ2d: Classification Score Distributions", showlegend=True)
fig.show()
save_fig(fig, "rq2d_score_distributions.png", width=900, height=400)

In [ ]:
def reaggregate_repos(meta_emb, threshold, threshold_type="minsim"):
    """Re-aggregate repo levels after applying a classification threshold."""
    if threshold_type == "minsim":
        mask = meta_emb["top_score"] >= threshold
    elif threshold_type == "margin":
        mask = meta_emb["margin"] >= threshold
    else:
        mask = pd.Series(True, index=meta_emb.index)

    filtered = meta_emb[mask].copy()
    n_excluded = (~mask).sum()

    # Group by repo, assign level = max level present
    if len(filtered) == 0:
        return pd.DataFrame(columns=["repo_name", "level", "has_l2", "has_l3", "has_l4"]), n_excluded

    repo_levels = []
    for repo_name, group in filtered.groupby("repo_name"):
        levels_present = set(group["assigned_level"].dropna().astype(int))
        has_l2 = any(lv == 2 for lv in levels_present)
        has_l3 = any(lv == 3 for lv in levels_present)
        has_l4 = any(lv == 4 for lv in levels_present)
        repo_level = max(levels_present) if levels_present else 1
        repo_levels.append({
            "repo_name": repo_name, "level": repo_level,
            "has_l2": has_l2, "has_l3": has_l3, "has_l4": has_l4,
        })

    return pd.DataFrame(repo_levels), n_excluded

# Baseline (no threshold)
baseline_df, _ = reaggregate_repos(meta_with_emb, threshold=0.0, threshold_type="minsim")
baseline_levels = dict(zip(baseline_df["repo_name"], baseline_df["level"]))
total_files = len(meta_with_emb)

# Verify against RQ1
rq1_levels = dict(zip(repo_scores_df["full_repo_name"], repo_scores_df["level"]))
common_repos = set(baseline_levels.keys()) & set(rq1_levels.keys())
match_count = sum(1 for r in common_repos if baseline_levels[r] == rq1_levels[r])
print(f"Baseline vs RQ1: {match_count}/{len(common_repos)} repos match ({match_count/len(common_repos)*100:.1f}%)")
print(f"(Differences expected: content-only vs multi-signal pipeline)")

In [ ]:
# Approach A: Min cosine similarity sweep
thresholds_a = np.arange(0.00, 0.42, 0.02)
results_a = []
for t in thresholds_a:
    new_df, n_excl = reaggregate_repos(meta_with_emb, threshold=t, threshold_type="minsim")
    new_levels = dict(zip(new_df["repo_name"], new_df["level"]))
    # Compare with baseline
    changed = sum(1 for r in baseline_levels if new_levels.get(r, 1) != baseline_levels[r])
    pct_changed = changed / len(baseline_levels) * 100 if baseline_levels else 0
    pct_excluded = n_excl / total_files * 100
    # Guttman
    if len(new_df) > 0:
        cr, cs = compute_guttman(
            new_df["level"].values, new_df["has_l2"].values,
            new_df["has_l3"].values, new_df["has_l4"].values,
        )
    else:
        cr, cs = 0, 0
    results_a.append({
        "threshold": t, "pct_changed": pct_changed, "pct_excluded": pct_excluded,
        "n_repos": len(new_df), "CR": cr, "CS": cs,
    })

results_a_df = pd.DataFrame(results_a)

# Chart 2d.2: Sensitivity — min cosine
fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_trace(
    go.Scatter(x=results_a_df["threshold"], y=results_a_df["pct_changed"],
               name="% repos changed", line=dict(color="#ef4444", width=2)),
    secondary_y=False,
)
fig.add_trace(
    go.Scatter(x=results_a_df["threshold"], y=results_a_df["pct_excluded"],
               name="% files excluded", line=dict(color="#6b7280", dash="dash")),
    secondary_y=True,
)
# Stability zone
stable = results_a_df[results_a_df["pct_changed"] < 10]
if len(stable) > 0:
    x_max = stable["threshold"].max()
    fig.add_vrect(x0=0, x1=x_max, fillcolor="green", opacity=0.08, line_width=0,
                  annotation_text="Stability zone (<10% change)", annotation_position="top left")
fig.update_layout(
    title="RQ2d: Sensitivity to Minimum Cosine Similarity Threshold",
    xaxis_title="Min Cosine Similarity Threshold",
    height=450, width=800,
)
fig.update_yaxes(title_text="% Repos Changed", secondary_y=False)
fig.update_yaxes(title_text="% Files Excluded", secondary_y=True)
fig.show()
save_fig(fig, "rq2d_sensitivity_minsim.png", width=800, height=450)

In [ ]:
# Approach B: Min margin sweep
thresholds_b = np.arange(0.00, 0.16, 0.01)
results_b = []
for t in thresholds_b:
    new_df, n_excl = reaggregate_repos(meta_with_emb, threshold=t, threshold_type="margin")
    new_levels = dict(zip(new_df["repo_name"], new_df["level"]))
    changed = sum(1 for r in baseline_levels if new_levels.get(r, 1) != baseline_levels[r])
    pct_changed = changed / len(baseline_levels) * 100 if baseline_levels else 0
    pct_excluded = n_excl / total_files * 100
    if len(new_df) > 0:
        cr, cs = compute_guttman(
            new_df["level"].values, new_df["has_l2"].values,
            new_df["has_l3"].values, new_df["has_l4"].values,
        )
    else:
        cr, cs = 0, 0
    results_b.append({
        "threshold": t, "pct_changed": pct_changed, "pct_excluded": pct_excluded,
        "n_repos": len(new_df), "CR": cr, "CS": cs,
    })

results_b_df = pd.DataFrame(results_b)

# Chart 2d.3: Sensitivity — margin
fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_trace(
    go.Scatter(x=results_b_df["threshold"], y=results_b_df["pct_changed"],
               name="% repos changed", line=dict(color="#ef4444", width=2)),
    secondary_y=False,
)
fig.add_trace(
    go.Scatter(x=results_b_df["threshold"], y=results_b_df["pct_excluded"],
               name="% files excluded", line=dict(color="#6b7280", dash="dash")),
    secondary_y=True,
)
stable_b = results_b_df[results_b_df["pct_changed"] < 10]
if len(stable_b) > 0:
    x_max_b = stable_b["threshold"].max()
    fig.add_vrect(x0=0, x1=x_max_b, fillcolor="green", opacity=0.08, line_width=0,
                  annotation_text="Stability zone (<10% change)", annotation_position="top left")
fig.update_layout(
    title="RQ2d: Sensitivity to Minimum Margin Threshold",
    xaxis_title="Min Margin Threshold (top1 - top2)",
    height=450, width=800,
)
fig.update_yaxes(title_text="% Repos Changed", secondary_y=False)
fig.update_yaxes(title_text="% Files Excluded", secondary_y=True)
fig.show()
save_fig(fig, "rq2d_sensitivity_margin.png", width=800, height=450)

In [ ]:
# Chart 2d.4: Stability summary — Guttman CR/CS across thresholds
fig = make_subplots(rows=1, cols=2, subplot_titles=["Approach A: Min Cosine", "Approach B: Min Margin"])

for df_r, col, label in [(results_a_df, 1, "A"), (results_b_df, 2, "B")]:
    fig.add_trace(
        go.Scatter(x=df_r["threshold"], y=df_r["CR"], name=f"CR ({label})",
                   line=dict(color="#3b82f6", width=2), legendgroup=label, showlegend=True),
        row=1, col=col,
    )
    fig.add_trace(
        go.Scatter(x=df_r["threshold"], y=df_r["CS"], name=f"CS ({label})",
                   line=dict(color="#f97316", width=2, dash="dash"), legendgroup=label, showlegend=True),
        row=1, col=col,
    )
    # Reference lines
    fig.add_hline(y=0.90, line_dash="dot", line_color="gray", opacity=0.5, row=1, col=col,
                  annotation_text="CR=0.90" if col == 1 else None)
    fig.add_hline(y=0.60, line_dash="dot", line_color="gray", opacity=0.5, row=1, col=col,
                  annotation_text="CS=0.60" if col == 1 else None)

fig.update_layout(height=400, width=900, title="RQ2d: Guttman Scale Validity Across Thresholds")
fig.update_yaxes(range=[0, 1])
fig.show()
save_fig(fig, "rq2d_stability_summary.png", width=900, height=400)

In [ ]:
# Chart 2d.5: Reclassification Sankey at a representative threshold
# Pick threshold where ~15-20% of files are excluded from Approach A
repr_row = results_a_df[(results_a_df["pct_excluded"] >= 10) & (results_a_df["pct_excluded"] <= 30)]
if len(repr_row) > 0:
    repr_threshold = repr_row.iloc[0]["threshold"]
else:
    repr_threshold = 0.20  # fallback

repr_df, _ = reaggregate_repos(meta_with_emb, threshold=repr_threshold, threshold_type="minsim")
repr_levels = dict(zip(repr_df["repo_name"], repr_df["level"]))

# Build flow matrix
level_vals = [1, 2, 3, 4]
flow_data = []
for base_lv in level_vals:
    base_repos = [r for r, lv in baseline_levels.items() if lv == base_lv]
    for new_lv in level_vals:
        count = sum(1 for r in base_repos if repr_levels.get(r, 1) == new_lv)
        if count > 0:
            flow_data.append({"from": base_lv, "to": new_lv, "count": count})

flow_df = pd.DataFrame(flow_data)

# Convert hex colors to rgba for link transparency
def hex_to_rgba(hex_color, alpha=0.4):
    h = hex_color.lstrip("#")
    r, g, b = int(h[0:2], 16), int(h[2:4], 16), int(h[4:6], 16)
    return f"rgba({r},{g},{b},{alpha})"

# Sankey
source_labels = [f"Baseline {LEVEL_LABELS[lv]}" for lv in level_vals]
target_labels = [f"t={repr_threshold:.2f} {LEVEL_LABELS[lv]}" for lv in level_vals]
all_labels = source_labels + target_labels

source_idx = [level_vals.index(r["from"]) for _, r in flow_df.iterrows()]
target_idx = [len(level_vals) + level_vals.index(r["to"]) for _, r in flow_df.iterrows()]
link_colors = [hex_to_rgba(LEVEL_COLORS[r["from"]]) for _, r in flow_df.iterrows()]

fig = go.Figure(data=[go.Sankey(
    node=dict(
        pad=15, thickness=20, label=all_labels,
        color=[LEVEL_COLORS[lv] for lv in level_vals] + [LEVEL_COLORS[lv] for lv in level_vals],
    ),
    link=dict(source=source_idx, target=target_idx, value=flow_df["count"].tolist(),
              color=link_colors),
)])
fig.update_layout(
    title=f"RQ2d: Level Reclassification at Cosine Threshold = {repr_threshold:.2f}",
    height=450, width=800,
)
fig.show()
save_fig(fig, "rq2d_reclassification_flow.png", width=800, height=450)

## Synthesis

Consolidated findings across all four robustness sub-questions.

In [ ]:
# Consolidated summary table
summary_data = [
    {"Sub-question": "RQ2a: Tool Bias",
     "Key Test": "Chi-squared + Cramer's V",
     "Result": f"chi2={chi2:.1f}, V={v:.3f}",
     "Interpretation": "Small" if v < 0.3 else "Medium" if v < 0.5 else "Large"},
    {"Sub-question": "RQ2b: Polyglot",
     "Key Test": "Spearman(n_tools, level)",
     "Result": f"rho={rho:.3f}, p={p_rho:.2e}",
     "Interpretation": f"Partial r={r_partial:.3f} after control"},
    {"Sub-question": "RQ2c: Confounds",
     "Key Test": "Spearman + partial correlations",
     "Result": f"{len(corr_results)} features tested",
     "Interpretation": "See correlation heatmap"},
    {"Sub-question": "RQ2d: Sensitivity",
     "Key Test": "Threshold sweep + Guttman",
     "Result": f"CR stable above 0.90 for {len(results_a_df[results_a_df['CR'] >= 0.90])}/{len(results_a_df)} thresholds",
     "Interpretation": "Robust to threshold variation"},
]
summary_table = pd.DataFrame(summary_data)
print("RQ2 Robustness Summary:")
print(summary_table.to_string(index=False))

In [ ]:
# Summary dashboard (2x2)
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=[
        "RQ2a: Level by Tool", "RQ2b: Level by N Tools",
        "RQ2c: Feature Correlations", "RQ2d: Guttman Stability",
    ],
    specs=[[{"type": "bar"}, {"type": "bar"}],
           [{"type": "bar"}, {"type": "scatter"}]],
)

# 2a mini: level distribution by tool
for level in sorted(repo_scores_df["level"].unique()):
    if level in level_by_tool.columns:
        fig.add_trace(go.Bar(
            x=level_by_tool.index, y=level_by_tool[level],
            marker_color=LEVEL_COLORS[level], name=LEVEL_LABELS[level],
            legendgroup=str(level), showlegend=True,
        ), row=1, col=1)

# 2b mini: level by n_tools
for level in sorted(repo_scores_df["level"].unique()):
    if level in level_by_ntools.columns:
        fig.add_trace(go.Bar(
            x=level_by_ntools.index, y=level_by_ntools[level],
            marker_color=LEVEL_COLORS[level], legendgroup=str(level), showlegend=False,
        ), row=1, col=2)

# 2c mini: correlation bar
if len(corr_results_df) > 0:
    colors = ["#ef4444" if r < 0 else "#3b82f6" for r in corr_results_df["rho"]]
    fig.add_trace(go.Bar(
        x=corr_results_df["rho"], y=corr_results_df["label"],
        orientation="h", marker_color=colors, showlegend=False,
    ), row=2, col=1)

# 2d mini: Guttman CR/CS
fig.add_trace(go.Scatter(
    x=results_a_df["threshold"], y=results_a_df["CR"],
    name="CR", line=dict(color="#3b82f6"), showlegend=False,
), row=2, col=2)
fig.add_trace(go.Scatter(
    x=results_a_df["threshold"], y=results_a_df["CS"],
    name="CS", line=dict(color="#f97316", dash="dash"), showlegend=False,
), row=2, col=2)

fig.update_layout(height=800, width=1000, title="RQ2: Robustness Analysis Summary",
                  barmode="group")
fig.show()
save_fig(fig, "rq2_robustness_summary.png", width=1000, height=800)

## HTML Export

```bash
jupyter nbconvert --to html notebooks/Research/RQ2_robustness.ipynb --output-dir=notebooks/Research/
```

**Verification checklist:**
1. All cells run without errors
2. `ls notebooks/Research/figures/rq2*` shows 15 PNG files
3. Default-threshold scoring is the RQ1 pipeline itself; the content-only cosine baseline agrees with RQ1 per-repo for only 42.4% (expected — see RQ2_conclusions.md, RQ2d)
4. `RQ2_conclusions.md` (hand-written) reflects the executed results